In [54]:
import os
import os.path as op
from collections import OrderedDict
import pandas as pd
import numpy as np
import shutil
import matplotlib.pyplot as plt
import seaborn as sns

In [55]:
# Define the main directory and target directory paths
deriv_dir = "./derivatives/none-reduced-motion"
reg_dir = os.path.join(deriv_dir, "regression")

## Create the dataframes for the Regression Analysis (physical health, rsFC)

#### Sig Dimensions:
##### Exluding the none network : Dim 1, 3


In [56]:
#make sure to consider if you want modified/orginial phy health variables
phyhealth_df = pd.read_csv(os.path.join(reg_dir, "phyhealth-reg-mod.csv"))

In [57]:
phyhealth_df

,src_subject_id,BMI,mctq_sdweek_calc,sleep_chrono,physical_activity1_y,cbcl_scr_syn_internal_t,cbcl_scr_syn_external_t,delta_weight,blood_pressure_mean,resp_composite
0,NDAR_INV030W95VP,30.209996,9.3677,neither,2.0,62.0,40.0,stable,79.166667,0.0
1,NDAR_INV0DC9BJZK,22.292121,6.7774,morning,7.0,63.0,34.0,stable,68.333333,0.0
2,NDAR_INV0DKWEM1A,30.716425,7.8534,morning,2.0,34.0,40.0,gain,91.555556,0.0
3,NDAR_INV0MPBK7TU,18.015334,8.8526,evening,3.0,52.0,51.0,stable,73.666667,0.0
4,NDAR_INV0RHLKA9M,23.193343,9.8785,neither,7.0,66.0,64.0,loss,91.833333,0.0
...,...,...,...,...,...,...,...,...,...,...
166,NDAR_INVZM2Y9JCA,13.754639,8.1008,neither,2.0,40.0,48.0,stable,73.444444,0.0
167,NDAR_INVZP49GXF4,20.982317,7.6798,evening,7.0,47.0,48.0,stable,73.555556,0.0
168,NDAR_INVZR9NMJBR,16.834671,9.0393,neither,5.0,44.0,34.0,stable,76.333333,0.0
169,NDAR_INVZT1J0KUC,22.657497,7.7963,neither,3.0,50.0,44.0,stable,93.166667,0.0


In [58]:
rsfc_df = pd.read_csv(os.path.join(deriv_dir, "rsfc-sub.csv"))
sociocult_df = pd.read_csv(os.path.join(deriv_dir, "sociocult_Nan.csv"))
covariate_df = pd.read_csv(os.path.join(deriv_dir, "covariate.csv"))


In [59]:
# we need to add back the subject id column so we remove the correct rows in the next step

# x is rsfc, y is sociocult
latent_df = pd.read_csv(op.join(deriv_dir, "rniXrsfc_lx-base.csv"))

latent_df["src_subject_id"] = rsfc_df["src_subject_id"].values

print(latent_df)

     Unnamed: 0            V1            V2            V3            V4  \
0             1  1.550499e-01 -5.447243e-04 -1.429859e-01  1.315287e-01   
1             2  9.727618e-03  6.511512e-02  2.434002e-01 -2.384341e-02   
2             3 -1.486726e-01  8.693889e-02 -7.536672e-03 -1.385905e-02   
3             4  5.248097e-02 -3.579174e-02  3.083647e-02  1.499342e-01   
4             5 -5.317479e-02 -1.450670e-01 -1.707414e-01  2.416174e-03   
..          ...           ...           ...           ...           ...   
231         232  2.486782e-01 -1.158487e-02  2.789390e-01  8.999250e-02   
232         233  1.339301e-15 -7.666418e-17 -1.196082e-15  1.296954e-15   
233         234  4.892722e-03  1.832317e-01  4.241857e-02 -9.560969e-02   
234         235  2.374475e-01  1.196338e-01  3.336592e-01 -1.050869e-01   
235         236  1.088294e-01  6.727357e-02 -8.428558e-02 -3.948547e-02   

               V5            V6            V7            V8            V9  \
0    9.905182e-02  6.7

In [60]:
latent_df = latent_df[latent_df["src_subject_id"].isin(phyhealth_df["src_subject_id"])]
sociocult_df = sociocult_df[sociocult_df["src_subject_id"].isin(phyhealth_df["src_subject_id"])]
covariate_df = covariate_df[covariate_df["src_subject_id"].isin(phyhealth_df["src_subject_id"])]
rsfc_df = rsfc_df[rsfc_df["src_subject_id"].isin(phyhealth_df["src_subject_id"])]

In [61]:
print(rsfc_df)

       src_subject_id  rsfmri_c_ngd_ad_ngd_ad  rsfmri_c_ngd_ad_ngd_cgc  \
1    NDAR_INV030W95VP                0.424266                 0.320737   
2    NDAR_INV0DC9BJZK                0.261305                 0.121087   
3    NDAR_INV0DKWEM1A                0.269920                 0.148276   
6    NDAR_INV0MPBK7TU                0.300166                 0.193031   
7    NDAR_INV0RHLKA9M                0.325872                 0.201443   
..                ...                     ...                      ...   
231  NDAR_INVZM2Y9JCA                0.227564                 0.064915   
232  NDAR_INVZP49GXF4                0.277297                 0.159869   
233  NDAR_INVZR9NMJBR                0.224798                 0.130237   
234  NDAR_INVZT1J0KUC                0.187419                 0.096684   
235  NDAR_INVZYRTFYRP                0.231318                 0.133982   

     rsfmri_c_ngd_ad_ngd_ca  rsfmri_c_ngd_ad_ngd_dt  rsfmri_c_ngd_ad_ngd_dla  \
1                  0.125510    

In [62]:
phyhealth_reg_save_path = os.path.join(reg_dir, "phyhealth-reg-mod.csv")
rsfc_reg_save_path = os.path.join(reg_dir, "rsfc-reg.csv")
latent_reg_save_path = os.path.join(reg_dir, "latent-reg.csv")
covariate_reg_save_path = os.path.join(reg_dir, "covariate-reg.csv")

phyhealth_df.to_csv(phyhealth_reg_save_path, index=False)
rsfc_df.to_csv(rsfc_reg_save_path, index=False)
latent_df.to_csv(latent_reg_save_path, index=False)
covariate_df.to_csv(covariate_reg_save_path, index=False)

print("All CSVs saved successfully!")

All CSVs saved successfully!


In [63]:
# create df for the corr coeff reg analysis
# rsfc measures that came back as signficant
# will make a df for each of these that has all of the covariates and phy health measures

# Define dimension 1 measures
dim1_rsfc_reg_measures = [
    #"rsfmri_c_ngd_cgc_ngd_dt",
    #"rsfmri_c_ngd_dt_ngd_dla",
    "rsfmri_c_ngd_dt_ngd_dt",
    "rsfmri_c_ngd_dt_ngd_vs",
    "rsfmri_c_ngd_vs_ngd_vs",
]
dim1_rsfc_short_labels = [
    #"cgc-dt", 
    #"dt-dla", 
    "DN-DN", 
    "DN-VN", 
    "VN-VN"]

# Define dimension 3 measures
dim3_rsfc_reg_measures = [
    "rsfmri_c_ngd_dt_ngd_smm",
    #"rsfmri_c_ngd_vta_ngd_vs",
]
dim3_rsfc_short_labels = [
    "DN-SMN", 
    #"vta-vs"
]

# Process dim1 measures
dim1_dir = op.join(reg_dir, "dim1")
os.makedirs(dim1_dir, exist_ok=True)

for measure, short_label in zip(dim1_rsfc_reg_measures, dim1_rsfc_short_labels):
    # 1) pull out subj‑ID + measure, then rename to "rsfc"
    tmp = rsfc_df[["src_subject_id", measure]].copy()
    tmp.rename(columns={measure: "rsfc"}, inplace=True)

    # 2) merge in covariates + phys‑health
    tmp = tmp.merge(covariate_df, on="src_subject_id", how="inner")
    tmp = tmp.merge(phyhealth_df, on="src_subject_id", how="inner")

    # 3) write out
    out_path = os.path.join(dim1_dir, f"phyhealth_{short_label}_data.csv")
    tmp.to_csv(out_path, index=False)
    print(f"Wrote {out_path}")

# Process dim3 measures
dim3_dir = op.join(reg_dir, "dim3")
os.makedirs(dim3_dir, exist_ok=True)

for measure, short_label in zip(dim3_rsfc_reg_measures, dim3_rsfc_short_labels):
    # 1) pull out subj‑ID + measure, then rename to "rsfc"
    tmp = rsfc_df[["src_subject_id", measure]].copy()
    tmp.rename(columns={measure: "rsfc"}, inplace=True)

    # 2) merge in covariates + phys‑health
    tmp = tmp.merge(covariate_df, on="src_subject_id", how="inner")
    tmp = tmp.merge(phyhealth_df, on="src_subject_id", how="inner")

    # 3) write out
    out_path = os.path.join(dim3_dir, f"phyhealth_{short_label}_data.csv")
    tmp.to_csv(out_path, index=False)
    print(f"Wrote {out_path}")

Wrote ./derivatives/none-reduced-motion/regression/dim1/phyhealth_DN-DN_data.csv
Wrote ./derivatives/none-reduced-motion/regression/dim1/phyhealth_DN-VN_data.csv
Wrote ./derivatives/none-reduced-motion/regression/dim1/phyhealth_VN-VN_data.csv
Wrote ./derivatives/none-reduced-motion/regression/dim3/phyhealth_DN-SMN_data.csv


In [64]:
# create df for the latent score reg analysis
# latent dimensions that came back as signficant
# will make a df for each of these that has all of the covariates and phy health measures

# Process dim1 latent scores
dim1_dir = op.join(reg_dir, "dim1")
os.makedirs(dim1_dir, exist_ok=True)

tmp = latent_df[["src_subject_id", "V1"]].copy()
tmp.rename(columns={"V1": "score"}, inplace=True)
tmp = tmp.merge(covariate_df, on="src_subject_id", how="inner")
tmp = tmp.merge(phyhealth_df, on="src_subject_id", how="inner")
out_path = os.path.join(dim1_dir, "phyhealth_dim1_latent_data.csv")
tmp.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

# Process dim3 latent scores
dim3_dir = op.join(reg_dir, "dim3")
os.makedirs(dim3_dir, exist_ok=True)

tmp = latent_df[["src_subject_id", "V3"]].copy()
tmp.rename(columns={"V3": "score"}, inplace=True)
tmp = tmp.merge(covariate_df, on="src_subject_id", how="inner")
tmp = tmp.merge(phyhealth_df, on="src_subject_id", how="inner")
out_path = os.path.join(dim3_dir, "phyhealth_dim3_latent_data.csv")
tmp.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

Wrote ./derivatives/none-reduced-motion/regression/dim1/phyhealth_dim1_latent_data.csv
Wrote ./derivatives/none-reduced-motion/regression/dim3/phyhealth_dim3_latent_data.csv
